# Fundus classification on Kaggle

Before running:
- Add the processed fundus dataset to the Notebook.
- Turn `Accelerator` on and select `GPU`.
- Turn `Internet` on so the Notebook can clone GitHub and download pretrained weights.
- The GitHub repository must contain `Source/training` and `Source/ensemble`.

In [ ]:
DATASET_SLUG = 'fundus-processed'
REPO_URL = 'https://github.com/nguyenquananh-2212/diabetic-retinopathy.git'
REPO_DIR = '/kaggle/working/diabetic-retinopathy'
DATA_ROOT = f'/kaggle/input/{DATASET_SLUG}/fundus_224_pad_v1'
WORK_CONFIG_DIR = '/kaggle/working/fundus_configs'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
RESULT_DIR = '/kaggle/working/results/ensemble_swin_efficientnet_v2_m'

RUN_SWIN = True
RUN_EFFICIENTNET = True
RUN_ENSEMBLE = True

print('DATA_ROOT:', DATA_ROOT)
print('REPO_DIR:', REPO_DIR)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)

In [ ]:
!nvidia-smi

In [ ]:
import torch
from pathlib import Path

NPROC = torch.cuda.device_count()
print('Detected GPUs:', NPROC)
if NPROC < 1:
    raise RuntimeError('No GPU detected. Enable GPU in Notebook settings.')

if not Path(DATA_ROOT).is_dir():
    raise FileNotFoundError(f'Dataset path does not exist: {DATA_ROOT}')
for split in ('train', 'val', 'test'):
    split_path = Path(DATA_ROOT) / split
    print(split, 'exists:', split_path.is_dir())

In [ ]:
import subprocess
import sys

repo_path = Path(REPO_DIR)
if not (repo_path / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

print('Repository ready:', REPO_DIR)
assert (repo_path / 'Source' / 'training' / 'train_swin.py').is_file()
assert (repo_path / 'Source' / 'training' / 'train_efficientnet_v2_m.py').is_file()
assert (repo_path / 'Source' / 'ensemble' / 'ensemble_test.py').is_file()

In [ ]:
!python -m pip install -q tqdm scikit-learn

In [ ]:
import json
from pathlib import Path

config_dir = Path(WORK_CONFIG_DIR)
config_dir.mkdir(parents=True, exist_ok=True)

def write_config(name, config):
    path = config_dir / name
    path.write_text(json.dumps(config, indent=2), encoding='utf-8')
    print(path)
    return path

common = {
    'data_root': DATA_ROOT,
    'num_classes': 5,
    'image_size': 224,
    'epochs': 20,
    'num_workers': 2,
    'learning_rate': 0.0001,
    'weight_decay': 0.0001,
    'patience': 5,
    'seed': 42,
    'pretrained': True,
    'amp': True,
    'resume': None,
}

swin_config = {**common, 'batch_size': 4, 'output_dir': f'{CHECKPOINT_DIR}/swin_t'}
efficientnet_config = {**common, 'batch_size': 4, 'output_dir': f'{CHECKPOINT_DIR}/efficientnet_v2_m'}
SWIN_CONFIG = write_config('swin_t_kaggle.json', swin_config)
EFFICIENTNET_CONFIG = write_config('efficientnet_v2_m_kaggle.json', efficientnet_config)

In [ ]:
if RUN_SWIN:
    subprocess.run([
        sys.executable,
        f'{REPO_DIR}/Source/training/train_swin.py',
        '--config',
        str(SWIN_CONFIG),
    ], check=True)

In [ ]:
if RUN_EFFICIENTNET:
    subprocess.run([
        sys.executable,
        f'{REPO_DIR}/Source/training/train_efficientnet_v2_m.py',
        '--config',
        str(EFFICIENTNET_CONFIG),
    ], check=True)

In [ ]:
ensemble_config = {
    'test_root': f'{DATA_ROOT}/test',
    'swin_checkpoint': f'{CHECKPOINT_DIR}/swin_t/best.pt',
    'efficientnet_checkpoint': f'{CHECKPOINT_DIR}/efficientnet_v2_m/best.pt',
    'output_dir': RESULT_DIR,
    'num_classes': 5,
    'image_size': 224,
    'batch_size': 4,
    'num_workers': 2,
    'swin_weight': 0.5,
    'efficientnet_weight': 0.5,
    'decision_rule': 'argmax',
    'amp': True,
}
ENSEMBLE_CONFIG = write_config('ensemble_kaggle.json', ensemble_config)

if RUN_ENSEMBLE:
    subprocess.run([
        sys.executable,
        f'{REPO_DIR}/Source/ensemble/ensemble_test.py',
        '--config',
        str(ENSEMBLE_CONFIG),
    ], check=True)

In [ ]:
from pathlib import Path

for path in [
    Path(f'{CHECKPOINT_DIR}/swin_t'),
    Path(f'{CHECKPOINT_DIR}/efficientnet_v2_m'),
    Path(RESULT_DIR),
]:
    if path.exists():
        print(f'\n--- {path} ---')
        for item in sorted(path.rglob('*')):
            if item.is_file():
                print(item)
